In [20]:
import cudf  # GPU-accelerated dataframe
import cupy as cp  # GPU-accelerated numerical computing
import numpy as np  # Numerical computing
import pandas as pd  # Fallback for CPU operations
import seaborn as sns

# Data Processing
from cuml.preprocessing import MinMaxScaler  # GPU-accelerated scaling
from sklearn.model_selection import train_test_split  # Fallback for data splitting

# Machine Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Evaluation Metrics
from cuml.metrics import regression
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Print library versions
print(f"cuDF Version: {cudf.__version__}")
print(f"CuPy Version: {cp.__version__}")
print(f"TensorFlow Version: {tf.__version__}")

# GPU Check
from numba import cuda

if cuda.is_available():
    print("\nGPU Detected:")
    device = cuda.get_current_device()
    print(f"Device Name: {device.name}") 
else:
    print("Warning: No GPU detected - Falling back to CPU mode")

cuDF Version: 24.10.01
CuPy Version: 13.3.0
TensorFlow Version: 2.17.0

GPU Detected:
Device Name: b'NVIDIA GeForce RTX 4050 Laptop GPU'


In [21]:
data_sample = pd.read_csv('/home/rna_13/ModelPredict/Data Prediksi/Data Beras Medium Normalisasi.csv', parse_dates=['Tanggal'], skipinitialspace=True)

In [22]:
data_sample_premium = pd.read_csv('/home/rna_13/ModelPredict/Data Prediksi/Data Beras Premium Normalisasi.csv', parse_dates=['Tanggal'], skipinitialspace=True)

In [23]:
data_sample_jagung = pd.read_csv('/home/rna_13/ModelPredict/Data Prediksi/Data Jagung Normalisasi.csv', parse_dates=['Tanggal'], skipinitialspace=True)

In [24]:
data_sample_kacang = pd.read_csv('/home/rna_13/ModelPredict/Data Prediksi/Data Kacang Hijau Normalisasi.csv', parse_dates=['Tanggal'], skipinitialspace=True)

In [25]:
data_sample

,Tanggal,Komoditi,Harga Petani,Harga Pengecer,Tipe,Periode,tahun,Luas Panen,Produktivitas,Produksi,...,harga_petani_scaled,harga_pengecer_scaled,luas_panen_scaled,produktivitas_scaled,produksi_scaled,hari_sin,hari_cos,bulan_sin,bulan_cos,tahun_norm
0,2022-01-01,Beras Medium,8799.791087,9000.002196,Padi,Q1-2022,2022.0,38556.6200,55.203588,212846.3758,...,0.363300,0.208314,1.000000,0.204491,1.000000,7.818315e-01,0.623490,5.000000e-01,0.866025,0.0
1,2022-01-02,Beras Medium,8799.789667,9000.002225,Padi,Q1-2022,2022.0,38556.6200,55.203588,212846.3758,...,0.363299,0.208314,1.000000,0.204491,1.000000,9.749279e-01,-0.222521,5.000000e-01,0.866025,0.0
2,2022-01-03,Beras Medium,8800.000000,9000.000000,Padi,Q1-2022,2022.0,38556.6200,55.203588,212846.3758,...,0.363425,0.208313,1.000000,0.204491,1.000000,4.338837e-01,-0.900969,5.000000e-01,0.866025,0.0
3,2022-01-04,Beras Medium,8800.000000,9000.000000,Padi,Q1-2022,2022.0,38556.6200,55.203588,212846.3758,...,0.363425,0.208313,1.000000,0.204491,1.000000,-4.338837e-01,-0.900969,5.000000e-01,0.866025,0.0
4,2022-01-05,Beras Medium,8800.000000,9000.000000,Padi,Q1-2022,2022.0,38556.6200,55.203588,212846.3758,...,0.363425,0.208313,1.000000,0.204491,1.000000,-9.749279e-01,-0.222521,5.000000e-01,0.866025,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
725,2023-12-27,Beras Medium,12500.000000,13800.000000,Padi,Q3-2023,2023.0,8101.2824,54.176608,43890.0000,...,2.585881,1.728352,0.038288,-0.022831,0.033388,-7.818315e-01,0.623490,-2.449294e-16,1.000000,1.0
726,2023-12-28,Beras Medium,12500.000000,13800.000000,Padi,Q3-2023,2023.0,8101.2824,54.176608,43890.0000,...,2.585881,1.728352,0.038288,-0.022831,0.033388,-9.797174e-16,1.000000,-2.449294e-16,1.000000,1.0
727,2023-12-29,Beras Medium,12500.000000,13800.000000,Padi,Q3-2023,2023.0,8101.2824,54.176608,43890.0000,...,2.585881,1.728352,0.038288,-0.022831,0.033388,7.818315e-01,0.623490,-2.449294e-16,1.000000,1.0
728,2023-12-30,Beras Medium,12500.000000,13800.000000,Padi,Q3-2023,2023.0,8101.2824,54.176608,43890.0000,...,2.585881,1.728352,0.038288,-0.022831,0.033388,9.749279e-01,-0.222521,-2.449294e-16,1.000000,1.0


In [26]:
import tensorflow as tf
from keras.models import load_model
import numpy as np # Untuk contoh prediksi

In [27]:
model_path = '/home/rna_13/ModelPredict/Model/ModelBerasMedium.keras' # Path yang sama saat menyimpan

try:
    loaded_model = load_model(model_path)
    print(f"\nModel berhasil dimuat ulang dari: {model_path}")
    loaded_model.summary() # Cek apakah arsitektur sama
except Exception as e:
    print(f"\nGagal memuat ulang model: {e}")
    loaded_model = None # Set None jika gagal


Model berhasil dimuat ulang dari: /home/rna_13/ModelPredict/Model/ModelBerasMedium.keras


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 128)        │        71,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 368,465 (1.41 MB)

 Trainable params: 122,821 (479.77 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 245,644 (959.55 KB)

In [28]:
model_path = '/home/rna_13/ModelPredict/Model/ModelBerasPremium.keras' # Path yang sama saat menyimpan

try:
    loaded_model_premium = load_model(model_path)
    print(f"\nModel berhasil dimuat ulang dari: {model_path}")
    loaded_model_premium.summary() # Cek apakah arsitektur sama
except Exception as e:
    print(f"\nGagal memuat ulang model: {e}")
    loaded_model_premium = None # Set None jika gagal


Model berhasil dimuat ulang dari: /home/rna_13/ModelPredict/Model/ModelBerasPremium.keras


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 128)        │        71,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 368,465 (1.41 MB)

 Trainable params: 122,821 (479.77 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 245,644 (959.55 KB)

In [29]:
model_path = '/home/rna_13/ModelPredict/Model/ModelJagung.keras' # Path yang sama saat menyimpan

try:
    loaded_model_jagung = load_model(model_path)
    print(f"\nModel berhasil dimuat ulang dari: {model_path}")
    loaded_model_jagung.summary() # Cek apakah arsitektur sama
except Exception as e:
    print(f"\nGagal memuat ulang model: {e}")
    loaded_model_jagung = None # Set None jika gagal


Model berhasil dimuat ulang dari: /home/rna_13/ModelPredict/Model/ModelJagung.keras


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 128)        │        71,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 368,465 (1.41 MB)

 Trainable params: 122,821 (479.77 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 245,644 (959.55 KB)

In [30]:
model_path = '/home/rna_13/ModelPredict/Model/ModelKacangHijau.keras' # Path yang sama saat menyimpan

try:
    loaded_model_kacang = load_model(model_path)
    print(f"\nModel berhasil dimuat ulang dari: {model_path}")
    loaded_model_kacang.summary() # Cek apakah arsitektur sama
except Exception as e:
    print(f"\nGagal memuat ulang model: {e}")
    loaded_model_kacang = None # Set None jika gagal


Model berhasil dimuat ulang dari: /home/rna_13/ModelPredict/Model/ModelKacangHijau.keras


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 128)        │        71,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 368,465 (1.41 MB)

 Trainable params: 122,821 (479.77 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 245,644 (959.55 KB)

In [31]:
import joblib
import os # For checking if the file exists

# Define the path to your saved scaler file
# Make sure this path is correct based on where you saved it!
scaler_file_path = 'scaler_beras_medium.pkl' 

# --- Load the scaler ---
if os.path.exists(scaler_file_path):
    # Load the scaler using joblib.load()
    loaded_scaler = joblib.load(scaler_file_path)
    print(f"Scaler successfully loaded from '{scaler_file_path}'")
    print(f"Type of loaded object: {type(loaded_scaler)}")
    # You can also check some attributes of the loaded scaler, e.g., if it's a MinMaxScaler
    # if hasattr(loaded_scaler, 'data_min_'):
    #     print(f"Scaler data min: {loaded_scaler.data_min_}")
else:
    print(f"Error: Scaler file not found at '{scaler_file_path}'.")
    print("Please ensure the file exists and the path is correct.")

Scaler successfully loaded from 'scaler_beras_medium.pkl'
Type of loaded object: <class 'cuml._thirdparty.sklearn.preprocessing._data.MinMaxScaler'>


In [32]:
import joblib
import os # For checking if the file exists

# Define the path to your saved scaler file
# Make sure this path is correct based on where you saved it!
scaler_file_path = '/home/rna_13/ModelPredict/Model/scaler_beras_premium.pkl' 

# --- Load the scaler ---
if os.path.exists(scaler_file_path):
    # Load the scaler using joblib.load()
    loaded_scaler_premium = joblib.load(scaler_file_path)
    print(f"Scaler successfully loaded from '{scaler_file_path}'")
    print(f"Type of loaded object: {type(loaded_scaler_premium)}")
    # You can also check some attributes of the loaded scaler, e.g., if it's a MinMaxScaler
    # if hasattr(loaded_scaler, 'data_min_'):
    #     print(f"Scaler data min: {loaded_scaler.data_min_}")
else:
    print(f"Error: Scaler file not found at '{scaler_file_path}'.")
    print("Please ensure the file exists and the path is correct.")

Scaler successfully loaded from '/home/rna_13/ModelPredict/Model/scaler_beras_premium.pkl'
Type of loaded object: <class 'cuml._thirdparty.sklearn.preprocessing._data.MinMaxScaler'>


In [47]:
# Define the path to your saved scaler file
# Make sure this path is correct based on where you saved it!
scaler_file_path = '/home/rna_13/ModelPredict/Model/scaler_jagung.pkl' 

# --- Load the scaler ---
if os.path.exists(scaler_file_path):
    # Load the scaler using joblib.load()
    loaded_scaler_jagung = joblib.load(scaler_file_path)
    print(f"Scaler successfully loaded from '{scaler_file_path}'")
    print(f"Type of loaded object: {type(loaded_scaler_jagung)}")
    # You can also check some attributes of the loaded scaler, e.g., if it's a MinMaxScaler
    # if hasattr(loaded_scaler, 'data_min_'):
    #     print(f"Scaler data min: {loaded_scaler.data_min_}")
else:
    print(f"Error: Scaler file not found at '{scaler_file_path}'.")
    print("Please ensure the file exists and the path is correct.")

Scaler successfully loaded from '/home/rna_13/ModelPredict/Model/scaler_jagung.pkl'
Type of loaded object: <class 'cuml._thirdparty.sklearn.preprocessing._data.MinMaxScaler'>


In [48]:
# Define the path to your saved scaler file
# Make sure this path is correct based on where you saved it!
scaler_file_path = '/home/rna_13/ModelPredict/Model/scaler_kacang_hijau.pkl' 

# --- Load the scaler ---
if os.path.exists(scaler_file_path):
    # Load the scaler using joblib.load()
    loaded_scaler_kacang = joblib.load(scaler_file_path)
    print(f"Scaler successfully loaded from '{scaler_file_path}'")
    print(f"Type of loaded object: {type(loaded_scaler_kacang)}")
    # You can also check some attributes of the loaded scaler, e.g., if it's a MinMaxScaler
    # if hasattr(loaded_scaler, 'data_min_'):
    #     print(f"Scaler data min: {loaded_scaler.data_min_}")
else:
    print(f"Error: Scaler file not found at '{scaler_file_path}'.")
    print("Please ensure the file exists and the path is correct.")


Scaler successfully loaded from '/home/rna_13/ModelPredict/Model/scaler_kacang_hijau.pkl'
Type of loaded object: <class 'cuml._thirdparty.sklearn.preprocessing._data.MinMaxScaler'>


In [35]:
feature_columns = [
    'harga_petani_scaled', 
    'harga_pengecer_scaled', 
    'luas_panen_scaled', 
    'produktivitas_scaled', 
    'produksi_scaled',       # sudah dinormalisasi
    'hari_sin',
    'hari_cos',
    'bulan_sin',
    'bulan_cos',
    'tahun_norm'
]

In [36]:
data_sample

,Tanggal,Komoditi,Harga Petani,Harga Pengecer,Tipe,Periode,tahun,Luas Panen,Produktivitas,Produksi,...,harga_petani_scaled,harga_pengecer_scaled,luas_panen_scaled,produktivitas_scaled,produksi_scaled,hari_sin,hari_cos,bulan_sin,bulan_cos,tahun_norm
0,2022-01-01,Beras Medium,8799.791087,9000.002196,Padi,Q1-2022,2022.0,38556.6200,55.203588,212846.3758,...,0.363300,0.208314,1.000000,0.204491,1.000000,7.818315e-01,0.623490,5.000000e-01,0.866025,0.0
1,2022-01-02,Beras Medium,8799.789667,9000.002225,Padi,Q1-2022,2022.0,38556.6200,55.203588,212846.3758,...,0.363299,0.208314,1.000000,0.204491,1.000000,9.749279e-01,-0.222521,5.000000e-01,0.866025,0.0
2,2022-01-03,Beras Medium,8800.000000,9000.000000,Padi,Q1-2022,2022.0,38556.6200,55.203588,212846.3758,...,0.363425,0.208313,1.000000,0.204491,1.000000,4.338837e-01,-0.900969,5.000000e-01,0.866025,0.0
3,2022-01-04,Beras Medium,8800.000000,9000.000000,Padi,Q1-2022,2022.0,38556.6200,55.203588,212846.3758,...,0.363425,0.208313,1.000000,0.204491,1.000000,-4.338837e-01,-0.900969,5.000000e-01,0.866025,0.0
4,2022-01-05,Beras Medium,8800.000000,9000.000000,Padi,Q1-2022,2022.0,38556.6200,55.203588,212846.3758,...,0.363425,0.208313,1.000000,0.204491,1.000000,-9.749279e-01,-0.222521,5.000000e-01,0.866025,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
725,2023-12-27,Beras Medium,12500.000000,13800.000000,Padi,Q3-2023,2023.0,8101.2824,54.176608,43890.0000,...,2.585881,1.728352,0.038288,-0.022831,0.033388,-7.818315e-01,0.623490,-2.449294e-16,1.000000,1.0
726,2023-12-28,Beras Medium,12500.000000,13800.000000,Padi,Q3-2023,2023.0,8101.2824,54.176608,43890.0000,...,2.585881,1.728352,0.038288,-0.022831,0.033388,-9.797174e-16,1.000000,-2.449294e-16,1.000000,1.0
727,2023-12-29,Beras Medium,12500.000000,13800.000000,Padi,Q3-2023,2023.0,8101.2824,54.176608,43890.0000,...,2.585881,1.728352,0.038288,-0.022831,0.033388,7.818315e-01,0.623490,-2.449294e-16,1.000000,1.0
728,2023-12-30,Beras Medium,12500.000000,13800.000000,Padi,Q3-2023,2023.0,8101.2824,54.176608,43890.0000,...,2.585881,1.728352,0.038288,-0.022831,0.033388,9.749279e-01,-0.222521,-2.449294e-16,1.000000,1.0


In [37]:
data_sample['Tanggal']

0     2022-01-01
1     2022-01-02
2     2022-01-03
3     2022-01-04
4     2022-01-05
         ...    
725   2023-12-27
726   2023-12-28
727   2023-12-29
728   2023-12-30
729   2023-12-31
Name: Tanggal, Length: 730, dtype: datetime64[ns]

In [38]:
data_sample['Tanggal'] = pd.to_datetime(data_sample['Tanggal'])

In [39]:
data_sample=data_sample.set_index('Tanggal')

In [40]:
scaled_data=data_sample[feature_columns]

In [41]:
scaled_data

,harga_petani_scaled,harga_pengecer_scaled,luas_panen_scaled,produktivitas_scaled,produksi_scaled,hari_sin,hari_cos,bulan_sin,bulan_cos,tahun_norm
Tanggal,,,,,,,,,,
2022-01-01,0.363300,0.208314,1.000000,0.204491,1.000000,7.818315e-01,0.623490,5.000000e-01,0.866025,0.0
2022-01-02,0.363299,0.208314,1.000000,0.204491,1.000000,9.749279e-01,-0.222521,5.000000e-01,0.866025,0.0
2022-01-03,0.363425,0.208313,1.000000,0.204491,1.000000,4.338837e-01,-0.900969,5.000000e-01,0.866025,0.0
2022-01-04,0.363425,0.208313,1.000000,0.204491,1.000000,-4.338837e-01,-0.900969,5.000000e-01,0.866025,0.0
2022-01-05,0.363425,0.208313,1.000000,0.204491,1.000000,-9.749279e-01,-0.222521,5.000000e-01,0.866025,0.0
...,...,...,...,...,...,...,...,...,...,...
2023-12-27,2.585881,1.728352,0.038288,-0.022831,0.033388,-7.818315e-01,0.623490,-2.449294e-16,1.000000,1.0
2023-12-28,2.585881,1.728352,0.038288,-0.022831,0.033388,-9.797174e-16,1.000000,-2.449294e-16,1.000000,1.0
2023-12-29,2.585881,1.728352,0.038288,-0.022831,0.033388,7.818315e-01,0.623490,-2.449294e-16,1.000000,1.0


In [42]:
data_sample.index[-1]

Timestamp('2023-12-31 00:00:00')

In [53]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

model=loaded_model
scaler=loaded_scaler

n_timesteps = 60  # Sesuaikan dengan model Anda

# Konversi scaled_data ke numpy array
if isinstance(scaled_data, pd.DataFrame):
    scaled_data_np = scaled_data.values
else:
    scaled_data_np = scaled_data

last_sequence = scaled_data_np[-n_timesteps:]
n_features = last_sequence.shape[1]  # Jumlah fitur total

# 2. Lakukan prediksi untuk N hari ke depan
days_ahead = 366
future_predictions = []

# Bentuk awal sequence untuk prediksi
current_seq = last_sequence.copy().reshape(1, n_timesteps, n_features)

for i in range(days_ahead):
    # Prediksi satu langkah ke depan
    pred_scaled = model.predict(current_seq, verbose=0)[0]  # Output: (n_output_features,)
    
    # Simpan prediksi
    future_predictions.append(pred_scaled)
    
    # Hitung tanggal prediksi untuk fitur temporal
    pred_date = data_sample.index[-1] + timedelta(days=i+1)
    
    # Hitung fitur temporal baru
    day_of_week = pred_date.weekday()
    month = pred_date.month
    year = pred_date.year
    
    # Hitung representasi siklis
    hari_sin = np.sin(2 * np.pi * day_of_week / 7)
    hari_cos = np.cos(2 * np.pi * day_of_week / 7)
    bulan_sin = np.sin(2 * np.pi * month / 12)
    bulan_cos = np.cos(2 * np.pi * month / 12)
    
    # Normalisasi tahun (gunakan min/max tahun dari data asli)
    tahun_min = data_sample.index.year.min()
    tahun_max = data_sample.index.year.max()
    tahun_norm = (year - tahun_min) / (tahun_max - tahun_min)
    
    # Gabungkan fitur prediksi dengan fitur temporal
    # Asumsikan model hanya memprediksi 4 fitur target
    full_features = np.concatenate([
        pred_scaled[:5],  # 4 fitur target
        [hari_sin, hari_cos, bulan_sin, bulan_cos, tahun_norm]  # 5 fitur temporal
    ])
    
    # Pastikan jumlah fitur sesuai
    if len(full_features) != n_features:
        # Jika tidak sesuai, sesuaikan dengan padding
        full_features = np.pad(
            full_features, 
            (0, n_features - len(full_features)), 
            'constant',
            constant_values=0
        )
    
    # Update sequence
    current_seq = np.concatenate([
        current_seq[:, 1:, :], 
        full_features.reshape(1, 1, n_features)
    ], axis=1)

# 3. Konversi hasil prediksi
future_predictions = np.array(future_predictions)

# 4. Denormalisasi hanya fitur target (5 fitur pertama)
future_predictions_denorm = scaler.inverse_transform(future_predictions[:, :5])

# 5. Buat DataFrame hasil prediksi
future_dates = [data_sample.index[-1] + timedelta(days=i+1) for i in range(days_ahead)]

df_predictions = pd.DataFrame({
    'Tanggal': future_dates,
    'harga_petani_pred': future_predictions_denorm[:, 0],
    'harga_pengecer_pred': future_predictions_denorm[:, 1],
    'luas_lahan_pred': future_predictions_denorm[:, 2],
    'produktivitas_pred': future_predictions_denorm[:, 3],
    'produksi_pred': future_predictions_denorm[:, 4]
})

# 6. Tambahkan informasi tambahan
df_predictions['hari'] = df_predictions['Tanggal'].dt.day_name()
df_predictions['bulan'] = df_predictions['Tanggal'].dt.month_name()
df_predictions['tahun'] = df_predictions['Tanggal'].dt.year

df_predictions['Komoditi']='Beras Medium'
df_pred_medium=df_predictions.copy()
print("Hasil Prediksi:")
print(df_pred_medium.head())

Hasil Prediksi:
     Tanggal  harga_petani_pred  harga_pengecer_pred  luas_lahan_pred  \
0 2024-01-01        9654.476562          9632.381836     32146.853516   
1 2024-01-02        9678.560547          9301.785156     32846.433594   
2 2024-01-03        9666.277344          9003.757812     33406.015625   
3 2024-01-04        9652.332031          8864.465820     33324.445312   
4 2024-01-05        9624.555664          8830.864258     32777.375000   

   produktivitas_pred  produksi_pred       hari    bulan  tahun      Komoditi  
0           55.341957  174017.437500     Monday  January   2024  Beras Medium  
1           55.442188  178129.765625    Tuesday  January   2024  Beras Medium  
2           55.465935  181458.765625  Wednesday  January   2024  Beras Medium  
3           55.467720  181873.781250   Thursday  January   2024  Beras Medium  
4           55.454090  179970.937500     Friday  January   2024  Beras Medium  


In [44]:
df_predictions

,Tanggal,harga_petani_pred,harga_pengecer_pred,luas_lahan_pred,produktivitas_pred,produksi_pred,hari,bulan,tahun
0,2024-01-01,9654.476562,9632.381836,32146.853516,55.341957,174017.437500,Monday,January,2024
1,2024-01-02,9678.560547,9301.785156,32846.433594,55.442188,178129.765625,Tuesday,January,2024
2,2024-01-03,9666.277344,9003.757812,33406.015625,55.465935,181458.765625,Wednesday,January,2024
3,2024-01-04,9652.332031,8864.465820,33324.445312,55.467720,181873.781250,Thursday,January,2024
4,2024-01-05,9624.555664,8830.864258,32777.375000,55.454090,179970.937500,Friday,January,2024
...,...,...,...,...,...,...,...,...,...
361,2024-12-27,9296.580078,10484.579102,27229.468750,55.278824,149013.656250,Friday,December,2024
362,2024-12-28,9334.701172,10608.944336,27493.595703,55.302967,152428.515625,Saturday,December,2024
363,2024-12-29,9389.016602,10789.737305,27904.923828,55.345783,156727.921875,Sunday,December,2024
364,2024-12-30,9415.270508,10878.391602,28233.156250,55.374992,158689.656250,Monday,December,2024


In [54]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

model=loaded_model_premium
scaler=loaded_scaler_premium

n_timesteps = 60  # Sesuaikan dengan model Anda

# Konversi scaled_data ke numpy array
if isinstance(scaled_data, pd.DataFrame):
    scaled_data_np = scaled_data.values
else:
    scaled_data_np = scaled_data

last_sequence = scaled_data_np[-n_timesteps:]
n_features = last_sequence.shape[1]  # Jumlah fitur total

# 2. Lakukan prediksi untuk N hari ke depan
days_ahead = 366
future_predictions = []

# Bentuk awal sequence untuk prediksi
current_seq = last_sequence.copy().reshape(1, n_timesteps, n_features)

for i in range(days_ahead):
    # Prediksi satu langkah ke depan
    pred_scaled = model.predict(current_seq, verbose=0)[0]  # Output: (n_output_features,)
    
    # Simpan prediksi
    future_predictions.append(pred_scaled)
    
    # Hitung tanggal prediksi untuk fitur temporal
    pred_date = data_sample.index[-1] + timedelta(days=i+1)
    
    # Hitung fitur temporal baru
    day_of_week = pred_date.weekday()
    month = pred_date.month
    year = pred_date.year
    
    # Hitung representasi siklis
    hari_sin = np.sin(2 * np.pi * day_of_week / 7)
    hari_cos = np.cos(2 * np.pi * day_of_week / 7)
    bulan_sin = np.sin(2 * np.pi * month / 12)
    bulan_cos = np.cos(2 * np.pi * month / 12)
    
    # Normalisasi tahun (gunakan min/max tahun dari data asli)
    tahun_min = data_sample.index.year.min()
    tahun_max = data_sample.index.year.max()
    tahun_norm = (year - tahun_min) / (tahun_max - tahun_min)
    
    # Gabungkan fitur prediksi dengan fitur temporal
    # Asumsikan model hanya memprediksi 4 fitur target
    full_features = np.concatenate([
        pred_scaled[:5],  # 4 fitur target
        [hari_sin, hari_cos, bulan_sin, bulan_cos, tahun_norm]  # 5 fitur temporal
    ])
    
    # Pastikan jumlah fitur sesuai
    if len(full_features) != n_features:
        # Jika tidak sesuai, sesuaikan dengan padding
        full_features = np.pad(
            full_features, 
            (0, n_features - len(full_features)), 
            'constant',
            constant_values=0
        )
    
    # Update sequence
    current_seq = np.concatenate([
        current_seq[:, 1:, :], 
        full_features.reshape(1, 1, n_features)
    ], axis=1)

# 3. Konversi hasil prediksi
future_predictions = np.array(future_predictions)

# 4. Denormalisasi hanya fitur target (5 fitur pertama)
future_predictions_denorm = scaler.inverse_transform(future_predictions[:, :5])

# 5. Buat DataFrame hasil prediksi
future_dates = [data_sample.index[-1] + timedelta(days=i+1) for i in range(days_ahead)]

df_predictions = pd.DataFrame({
    'Tanggal': future_dates,
    'harga_petani_pred': future_predictions_denorm[:, 0],
    'harga_pengecer_pred': future_predictions_denorm[:, 1],
    'luas_lahan_pred': future_predictions_denorm[:, 2],
    'produktivitas_pred': future_predictions_denorm[:, 3],
    'produksi_pred': future_predictions_denorm[:, 4]
})

# 6. Tambahkan informasi tambahan
df_predictions['hari'] = df_predictions['Tanggal'].dt.day_name()
df_predictions['bulan'] = df_predictions['Tanggal'].dt.month_name()
df_predictions['tahun'] = df_predictions['Tanggal'].dt.year

df_predictions['Komoditi']='Beras Premium'
df_pred_premium=df_predictions.copy()
print("Hasil Prediksi:")
print(df_pred_premium.head())

Hasil Prediksi:
     Tanggal  harga_petani_pred  harga_pengecer_pred  luas_lahan_pred  \
0 2024-01-01       10073.138672         11164.091797     28211.816406   
1 2024-01-02       10068.591797         11154.531250     28356.291016   
2 2024-01-03       10057.289062         11134.898438     28428.263672   
3 2024-01-04       10042.980469         11107.096680     28523.548828   
4 2024-01-05       10031.195312         11094.952148     28475.488281   

   produktivitas_pred  produksi_pred       hari    bulan  tahun       Komoditi  
0           55.268192  154993.500000     Monday  January   2024  Beras Premium  
1           55.253952  155678.187500    Tuesday  January   2024  Beras Premium  
2           55.239906  156181.578125  Wednesday  January   2024  Beras Premium  
3           55.231907  156599.781250   Thursday  January   2024  Beras Premium  
4           55.226910  156342.578125     Friday  January   2024  Beras Premium  


In [55]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

model=loaded_model_jagung
scaler=loaded_scaler_jagung

n_timesteps = 60  # Sesuaikan dengan model Anda

# Konversi scaled_data ke numpy array
if isinstance(scaled_data, pd.DataFrame):
    scaled_data_np = scaled_data.values
else:
    scaled_data_np = scaled_data

last_sequence = scaled_data_np[-n_timesteps:]
n_features = last_sequence.shape[1]  # Jumlah fitur total

# 2. Lakukan prediksi untuk N hari ke depan
days_ahead = 366
future_predictions = []

# Bentuk awal sequence untuk prediksi
current_seq = last_sequence.copy().reshape(1, n_timesteps, n_features)

for i in range(days_ahead):
    # Prediksi satu langkah ke depan
    pred_scaled = model.predict(current_seq, verbose=0)[0]  # Output: (n_output_features,)
    
    # Simpan prediksi
    future_predictions.append(pred_scaled)
    
    # Hitung tanggal prediksi untuk fitur temporal
    pred_date = data_sample.index[-1] + timedelta(days=i+1)
    
    # Hitung fitur temporal baru
    day_of_week = pred_date.weekday()
    month = pred_date.month
    year = pred_date.year
    
    # Hitung representasi siklis
    hari_sin = np.sin(2 * np.pi * day_of_week / 7)
    hari_cos = np.cos(2 * np.pi * day_of_week / 7)
    bulan_sin = np.sin(2 * np.pi * month / 12)
    bulan_cos = np.cos(2 * np.pi * month / 12)
    
    # Normalisasi tahun (gunakan min/max tahun dari data asli)
    tahun_min = data_sample.index.year.min()
    tahun_max = data_sample.index.year.max()
    tahun_norm = (year - tahun_min) / (tahun_max - tahun_min)
    
    # Gabungkan fitur prediksi dengan fitur temporal
    # Asumsikan model hanya memprediksi 4 fitur target
    full_features = np.concatenate([
        pred_scaled[:5],  # 4 fitur target
        [hari_sin, hari_cos, bulan_sin, bulan_cos, tahun_norm]  # 5 fitur temporal
    ])
    
    # Pastikan jumlah fitur sesuai
    if len(full_features) != n_features:
        # Jika tidak sesuai, sesuaikan dengan padding
        full_features = np.pad(
            full_features, 
            (0, n_features - len(full_features)), 
            'constant',
            constant_values=0
        )
    
    # Update sequence
    current_seq = np.concatenate([
        current_seq[:, 1:, :], 
        full_features.reshape(1, 1, n_features)
    ], axis=1)

# 3. Konversi hasil prediksi
future_predictions = np.array(future_predictions)

# 4. Denormalisasi hanya fitur target (5 fitur pertama)
future_predictions_denorm = scaler.inverse_transform(future_predictions[:, :5])

# 5. Buat DataFrame hasil prediksi
future_dates = [data_sample.index[-1] + timedelta(days=i+1) for i in range(days_ahead)]

df_predictions = pd.DataFrame({
    'Tanggal': future_dates,
    'harga_petani_pred': future_predictions_denorm[:, 0],
    'harga_pengecer_pred': future_predictions_denorm[:, 1],
    'luas_lahan_pred': future_predictions_denorm[:, 2],
    'produktivitas_pred': future_predictions_denorm[:, 3],
    'produksi_pred': future_predictions_denorm[:, 4]
})

# 6. Tambahkan informasi tambahan
df_predictions['hari'] = df_predictions['Tanggal'].dt.day_name()
df_predictions['bulan'] = df_predictions['Tanggal'].dt.month_name()
df_predictions['tahun'] = df_predictions['Tanggal'].dt.year

df_predictions['Komoditi']='Jagung'
df_pred_jagung=df_predictions.copy()
print("Hasil Prediksi:")
print(df_pred_jagung.head())

Hasil Prediksi:
     Tanggal  harga_petani_pred  harga_pengecer_pred  luas_lahan_pred  \
0 2024-01-01        3943.283203          4356.760254     13853.442383   
1 2024-01-02        3984.018555          4340.216309     22928.005859   
2 2024-01-03        3977.182129          4308.191895     35320.042969   
3 2024-01-04        3929.395020          4280.340332     45749.656250   
4 2024-01-05        3869.668945          4268.393066     52700.308594   

   produktivitas_pred  produksi_pred       hari    bulan  tahun Komoditi  
0           69.503563  163219.500000     Monday  January   2024   Jagung  
1           69.837257  220739.890625    Tuesday  January   2024   Jagung  
2           70.248260  305990.531250  Wednesday  January   2024   Jagung  
3           70.489441  371805.312500   Thursday  January   2024   Jagung  
4           70.567795  416294.687500     Friday  January   2024   Jagung  


In [56]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

model=loaded_model_kacang
scaler=loaded_scaler_kacang

n_timesteps = 60  # Sesuaikan dengan model Anda

# Konversi scaled_data ke numpy array
if isinstance(scaled_data, pd.DataFrame):
    scaled_data_np = scaled_data.values
else:
    scaled_data_np = scaled_data

last_sequence = scaled_data_np[-n_timesteps:]
n_features = last_sequence.shape[1]  # Jumlah fitur total

# 2. Lakukan prediksi untuk N hari ke depan
days_ahead = 366
future_predictions = []

# Bentuk awal sequence untuk prediksi
current_seq = last_sequence.copy().reshape(1, n_timesteps, n_features)

for i in range(days_ahead):
    # Prediksi satu langkah ke depan
    pred_scaled = model.predict(current_seq, verbose=0)[0]  # Output: (n_output_features,)
    
    # Simpan prediksi
    future_predictions.append(pred_scaled)
    
    # Hitung tanggal prediksi untuk fitur temporal
    pred_date = data_sample.index[-1] + timedelta(days=i+1)
    
    # Hitung fitur temporal baru
    day_of_week = pred_date.weekday()
    month = pred_date.month
    year = pred_date.year
    
    # Hitung representasi siklis
    hari_sin = np.sin(2 * np.pi * day_of_week / 7)
    hari_cos = np.cos(2 * np.pi * day_of_week / 7)
    bulan_sin = np.sin(2 * np.pi * month / 12)
    bulan_cos = np.cos(2 * np.pi * month / 12)
    
    # Normalisasi tahun (gunakan min/max tahun dari data asli)
    tahun_min = data_sample.index.year.min()
    tahun_max = data_sample.index.year.max()
    tahun_norm = (year - tahun_min) / (tahun_max - tahun_min)
    
    # Gabungkan fitur prediksi dengan fitur temporal
    # Asumsikan model hanya memprediksi 4 fitur target
    full_features = np.concatenate([
        pred_scaled[:5],  # 4 fitur target
        [hari_sin, hari_cos, bulan_sin, bulan_cos, tahun_norm]  # 5 fitur temporal
    ])
    
    # Pastikan jumlah fitur sesuai
    if len(full_features) != n_features:
        # Jika tidak sesuai, sesuaikan dengan padding
        full_features = np.pad(
            full_features, 
            (0, n_features - len(full_features)), 
            'constant',
            constant_values=0
        )
    
    # Update sequence
    current_seq = np.concatenate([
        current_seq[:, 1:, :], 
        full_features.reshape(1, 1, n_features)
    ], axis=1)

# 3. Konversi hasil prediksi
future_predictions = np.array(future_predictions)

# 4. Denormalisasi hanya fitur target (5 fitur pertama)
future_predictions_denorm = scaler.inverse_transform(future_predictions[:, :5])

# 5. Buat DataFrame hasil prediksi
future_dates = [data_sample.index[-1] + timedelta(days=i+1) for i in range(days_ahead)]

df_predictions = pd.DataFrame({
    'Tanggal': future_dates,
    'harga_petani_pred': future_predictions_denorm[:, 0],
    'harga_pengecer_pred': future_predictions_denorm[:, 1],
    'luas_lahan_pred': future_predictions_denorm[:, 2],
    'produktivitas_pred': future_predictions_denorm[:, 3],
    'produksi_pred': future_predictions_denorm[:, 4]
})

# 6. Tambahkan informasi tambahan
df_predictions['hari'] = df_predictions['Tanggal'].dt.day_name()
df_predictions['bulan'] = df_predictions['Tanggal'].dt.month_name()
df_predictions['tahun'] = df_predictions['Tanggal'].dt.year

df_predictions['Komoditi']='Kacang Hijau'
df_pred_kacang=df_predictions.copy()
print("Hasil Prediksi:")
print(df_pred_kacang.head())

Hasil Prediksi:
     Tanggal  harga_petani_pred  harga_pengecer_pred  luas_lahan_pred  \
0 2024-01-01       16547.787109         18622.500000      -842.144165   
1 2024-01-02       15615.841797         17949.583984      -872.835571   
2 2024-01-03       14394.878906         17329.490234      -775.088135   
3 2024-01-04       13338.458008         16656.410156     -1107.580078   
4 2024-01-05       13017.676758         16117.931641      -610.971130   

   produktivitas_pred  produksi_pred       hari    bulan  tahun      Komoditi  
0           -0.653473    -795.964905     Monday  January   2024  Kacang Hijau  
1           -0.863882   -1093.055298    Tuesday  January   2024  Kacang Hijau  
2           -1.029089   -1230.383667  Wednesday  January   2024  Kacang Hijau  
3           -0.702285   -1121.534912   Thursday  January   2024  Kacang Hijau  
4           -0.469498    -783.318054     Friday  January   2024  Kacang Hijau  


In [61]:
df_merged_beras = pd.concat([df_pred_medium, df_pred_premium], ignore_index=True)
    

In [62]:
df_merged_beras

,Tanggal,harga_petani_pred,harga_pengecer_pred,luas_lahan_pred,produktivitas_pred,produksi_pred,hari,bulan,tahun,Komoditi
0,2024-01-01,9654.476562,9632.381836,32146.853516,55.341957,174017.437500,Monday,January,2024,Beras Medium
1,2024-01-02,9678.560547,9301.785156,32846.433594,55.442188,178129.765625,Tuesday,January,2024,Beras Medium
2,2024-01-03,9666.277344,9003.757812,33406.015625,55.465935,181458.765625,Wednesday,January,2024,Beras Medium
3,2024-01-04,9652.332031,8864.465820,33324.445312,55.467720,181873.781250,Thursday,January,2024,Beras Medium
4,2024-01-05,9624.555664,8830.864258,32777.375000,55.454090,179970.937500,Friday,January,2024,Beras Medium
...,...,...,...,...,...,...,...,...,...,...
727,2024-12-27,10024.545898,11160.468750,27428.244141,55.221664,152372.750000,Friday,December,2024,Beras Premium
728,2024-12-28,10021.978516,11151.132812,27298.068359,55.220139,151648.281250,Saturday,December,2024,Beras Premium
729,2024-12-29,10026.064453,11156.355469,27245.156250,55.218498,151406.781250,Sunday,December,2024,Beras Premium
730,2024-12-30,10034.578125,11174.479492,27319.498047,55.218609,151897.250000,Monday,December,2024,Beras Premium


In [63]:
df_merged_kacang=pd.concat([df_pred_jagung, df_pred_kacang], ignore_index=True)

In [72]:
df_merged=pd.concat([df_merged_beras, df_merged_kacang], ignore_index=True)

In [73]:
df_merged

,Tanggal,harga_petani_pred,harga_pengecer_pred,luas_lahan_pred,produktivitas_pred,produksi_pred,hari,bulan,tahun,Komoditi
0,2024-01-01,9654.476562,9632.381836,32146.853516,55.341957,174017.437500,Monday,January,2024,Beras Medium
1,2024-01-02,9678.560547,9301.785156,32846.433594,55.442188,178129.765625,Tuesday,January,2024,Beras Medium
2,2024-01-03,9666.277344,9003.757812,33406.015625,55.465935,181458.765625,Wednesday,January,2024,Beras Medium
3,2024-01-04,9652.332031,8864.465820,33324.445312,55.467720,181873.781250,Thursday,January,2024,Beras Medium
4,2024-01-05,9624.555664,8830.864258,32777.375000,55.454090,179970.937500,Friday,January,2024,Beras Medium
...,...,...,...,...,...,...,...,...,...,...
1459,2024-12-27,12948.521484,17288.531250,-23.731987,-0.185413,-68.628693,Friday,December,2024,Kacang Hijau
1460,2024-12-28,12948.897461,17262.468750,-19.102840,-0.143656,-39.113747,Saturday,December,2024,Kacang Hijau
1461,2024-12-29,12948.719727,17292.187500,-20.651508,-0.137412,-25.448574,Sunday,December,2024,Kacang Hijau
1462,2024-12-30,12947.764648,17361.931641,-29.908911,-0.171909,-38.210197,Monday,December,2024,Kacang Hijau


In [74]:
df_merged=df_merged.set_index('Tanggal')
df_merged=df_merged.sort_values(by='Tanggal')


In [75]:
df_merged

,harga_petani_pred,harga_pengecer_pred,luas_lahan_pred,produktivitas_pred,produksi_pred,hari,bulan,tahun,Komoditi
Tanggal,,,,,,,,,
2024-01-01,9654.476562,9632.381836,32146.853516,55.341957,174017.437500,Monday,January,2024,Beras Medium
2024-01-01,3943.283203,4356.760254,13853.442383,69.503563,163219.500000,Monday,January,2024,Jagung
2024-01-01,16547.787109,18622.500000,-842.144165,-0.653473,-795.964905,Monday,January,2024,Kacang Hijau
2024-01-01,10073.138672,11164.091797,28211.816406,55.268192,154993.500000,Monday,January,2024,Beras Premium
2024-01-02,9678.560547,9301.785156,32846.433594,55.442188,178129.765625,Tuesday,January,2024,Beras Medium
...,...,...,...,...,...,...,...,...,...
2024-12-30,9415.270508,10878.391602,28233.156250,55.374992,158689.656250,Monday,December,2024,Beras Medium
2024-12-31,10040.048828,11188.854492,27435.550781,55.219128,152524.812500,Tuesday,December,2024,Beras Premium
2024-12-31,9400.598633,10827.897461,28392.152344,55.362995,157844.484375,Tuesday,December,2024,Beras Medium


In [77]:
path='/home/rna_13/ModelPredict/Data Prediksi/Data Hasil Prediksi 2024.csv'

In [81]:
df=pd.read_csv('/home/rna_13/ModelPredict/Data Prediksi/Data Hasil Prediksi 2024.csv', parse_dates=['Tanggal'], skipinitialspace=True)


In [139]:
df_lengkap=pd.read_csv('/home/rna_13/ModelPredict/harga.csv', parse_dates=['Tanggal'], skipinitialspace=True)

In [141]:
df_lengkap=df_lengkap.sort_values('Tanggal')

In [142]:
df_lengkap

,Komoditi,Tanggal,Harga Petani,Harga Pengecer
1063,Beras Medium,2022-01-01,NaN,NaN
2128,Beras Premium,2022-01-01,NaN,NaN
3193,Kacang Hijau,2022-01-01,NaN,NaN
2129,Beras Premium,2022-01-02,NaN,NaN
1064,Beras Medium,2022-01-02,NaN,NaN
...,...,...,...,...
4380,Jagung Pipil Kering,2024-12-30,NaN,NaN
4319,Beras Medium,2024-12-31,NaN,NaN
4288,Kacang Hijau,2024-12-31,NaN,NaN
4350,Beras Premium,2024-12-31,NaN,NaN


In [143]:
df_lengkap.reset_index(drop=True, inplace=True)

In [144]:
df_lengkap['Komoditi'].unique()

array(['Beras Medium', 'Beras Premium', 'Kacang Hijau',
       'Jagung Pipil Kering'], dtype=object)

In [166]:
df_aktual = df_lengkap[(df_lengkap['Tanggal'] >= '2024-01-01') & (df_lengkap['Tanggal'] <= '2024-12-31')].copy()

In [167]:
df_aktual=df_aktual.sort_values('Tanggal')
df_aktual=df_aktual.reset_index()
df_aktual=df_aktual.drop(columns=['index'])
df_aktual

,Komoditi,Tanggal,Harga Petani,Harga Pengecer
0,Kacang Hijau,2024-01-01,NaN,NaN
1,Beras Premium,2024-01-01,NaN,NaN
2,Beras Medium,2024-01-01,NaN,NaN
3,Jagung Pipil Kering,2024-01-01,NaN,NaN
4,Beras Medium,2024-01-02,12500.0,13800.0
...,...,...,...,...
1459,Beras Medium,2024-12-30,NaN,NaN
1460,Kacang Hijau,2024-12-31,NaN,NaN
1461,Beras Premium,2024-12-31,NaN,NaN
1462,Beras Medium,2024-12-31,NaN,NaN


In [154]:
df_aktual = df_aktual.rename(columns={
    ' Harga Petani ': 'Harga Petani',
    ' Harga Pengecer ': 'Harga Pengecer'
})

In [155]:
kolom_harga=[
    'Tanggal',
    'Harga Petani ', 
    'Harga Pengecer '
]

In [105]:
df_aktual.dtypes

Komoditi                   object
Tanggal            datetime64[ns]
Harga Petani              float64
Harga Pengecer            float64
dtype: object

In [156]:
df_aktual

,Komoditi,Tanggal,Harga Petani,Harga Pengecer
0,Kacang Hijau,2024-01-01,NaN,NaN
1,Beras Premium,2024-01-01,NaN,NaN
2,Beras Medium,2024-01-01,NaN,NaN
3,Jagung Pipil Kering,2024-01-01,NaN,NaN
4,Beras Medium,2024-01-02,12500.0,13800.0
...,...,...,...,...
1459,Beras Medium,2024-12-30,NaN,NaN
1460,Kacang Hijau,2024-12-31,NaN,NaN
1461,Beras Premium,2024-12-31,NaN,NaN
1462,Beras Medium,2024-12-31,NaN,NaN


In [107]:
testppp=df_aktual['Harga Pengecer ']

In [157]:
df_aktual = df_aktual[kolom_harga]

In [90]:
from statsmodels.tsa.seasonal import STL

In [158]:
def stl_imputation(series, period=30):
    """
    Mengisi missing value (NaN) dalam deret waktu menggunakan dekomposisi STL.
    
    Argumen:
    series (pd.Series): Deret waktu dengan kemungkinan missing value (NaN).
    period (int): Periode musiman deret waktu (misal 30 untuk data harian bulanan).

    Mengembalikan:
    pd.Series: Deret waktu dengan missing value yang sudah diisi.
    """
    
    # Pastikan series adalah Pandas Series dan indeksnya berupa datetime
    if not isinstance(series, pd.Series):
        series = pd.Series(series)
    
    # Buat salinan seri untuk bekerja, agar tidak mengubah seri asli
    imputed_series = series.copy()
    
    # Identifikasi indeks missing values
    missing_indices = imputed_series[imputed_series.isnull()].index
    
    if missing_indices.empty:
        print("Tidak ada missing value untuk diisi.")
        return series
        
    print(f"Mengisi {len(missing_indices)} missing value menggunakan STL imputation...")

    # Langkah 1: Interpolasi awal untuk mengisi NaN sementara.
    # STL tidak bisa bekerja dengan NaN di tengah seri.
    # Interpolasi linear adalah pilihan umum untuk ini.
    temp_series = imputed_series.interpolate(method='linear', limit_direction='both')
    
    # Jika masih ada NaN di awal/akhir setelah interpolasi, isi dengan mean
    if temp_series.isnull().any():
        temp_series = temp_series.fillna(temp_series.mean())

    # Langkah 2: Dekomposisi STL
    # Pastikan period sesuai dengan data Anda (misal 30 untuk bulanan)
    stl = STL(temp_series, period=period, robust=True)
    result = stl.fit()
    
    # Langkah 3: Rekonstruksi tanpa noise (menggunakan trend + seasonal)
    reconstructed_values = result.trend + result.seasonal
    
    # Langkah 4: Isi missing value dengan nilai rekonstruksi
    # Kita hanya mengganti nilai di indeks yang tadinya NaN
    imputed_series.loc[missing_indices] = reconstructed_values.loc[missing_indices]
    
    print("Pengisian missing value selesai.")
    return imputed_series

In [165]:
df_aktual

,Tanggal,Harga Petani,Harga Pengecer
0,2024-01-01,12625.736548,13781.106934
1,2024-01-01,12430.783214,13686.647524
2,2024-01-01,12467.936563,14042.358097
3,2024-01-01,12558.242739,13760.337283
4,2024-01-02,12500.000000,13800.000000
...,...,...,...
1459,2024-12-30,12908.098000,14806.059814
1460,2024-12-31,12931.576593,14869.377858
1461,2024-12-31,12991.255415,14973.538073
1462,2024-12-31,13057.920810,15118.036751


In [168]:
df_jagung = df_aktual[df_aktual['Komoditi'] == 'Jagung Pipil Kering']
df_kacang = df_aktual[df_aktual['Komoditi'] == 'Kacang Hijau']
df_medium= df_aktual[df_aktual['Komoditi'] == 'Beras Medium']
df_premium= df_aktual[df_aktual['Komoditi'] == 'Beras Premium']

In [169]:
df_list=[
    df_medium,
    df_premium,
    df_jagung,
    df_kacang
]

In [170]:
for dfk in df_list:
    dfk['Harga Petani '] = stl_imputation(dfk['Harga Petani '], period=30) # Asumsi periode mingguan
    dfk['Harga Pengecer '] = stl_imputation(dfk['Harga Pengecer '], period=30)

Mengisi 113 missing value menggunakan STL imputation...
Pengisian missing value selesai.
Mengisi 113 missing value menggunakan STL imputation...
Pengisian missing value selesai.
Mengisi 114 missing value menggunakan STL imputation...
Pengisian missing value selesai.
Mengisi 114 missing value menggunakan STL imputation...
Pengisian missing value selesai.
Mengisi 114 missing value menggunakan STL imputation...
Pengisian missing value selesai.
Mengisi 114 missing value menggunakan STL imputation...
Pengisian missing value selesai.
Mengisi 114 missing value menggunakan STL imputation...
Pengisian missing value selesai.
Mengisi 114 missing value menggunakan STL imputation...
Pengisian missing value selesai.


In [171]:
for dfk in df_list:
    print("\nDataFrame setelah imputasi:")
    print(dfk)
    print("\nMissing values setelah imputasi:")
    print(dfk['Harga Petani '].isnull().sum())
    print(dfk['Harga Pengecer '].isnull().sum())


DataFrame setelah imputasi:
          Komoditi    Tanggal  Harga Petani   Harga Pengecer 
2     Beras Medium 2024-01-01   12468.324314     13798.906306
4     Beras Medium 2024-01-02   12500.000000     13800.000000
11    Beras Medium 2024-01-03   12500.000000     13800.000000
14    Beras Medium 2024-01-04   12500.000000     13800.000000
18    Beras Medium 2024-01-05   12500.000000     13800.000000
...            ...        ...            ...              ...
1445  Beras Medium 2024-12-27   12500.000000     13500.000000
1449  Beras Medium 2024-12-28   12500.000000     13500.000000
1454  Beras Medium 2024-12-29   12500.000000     13500.000000
1459  Beras Medium 2024-12-30   12480.126658     13501.048504
1462  Beras Medium 2024-12-31   12532.685674     13504.871653

[366 rows x 4 columns]

Missing values setelah imputasi:
0
0

DataFrame setelah imputasi:
           Komoditi    Tanggal  Harga Petani   Harga Pengecer 
1     Beras Premium 2024-01-01   13513.085957     14589.632582
7     Bera

In [159]:
print("DataFrame sebelum imputasi:")
print(df_aktual)
print("\nMissing values sebelum imputasi:")
print(df_aktual['Harga Petani '].isnull().sum())
print(df_aktual['Harga Pengecer '].isnull().sum())


# Pastikan kolom 'Tanggal' adalah indeks

# Terapkan fungsi stl_imputation ke kolom 'Harga Petani'
# Karena STL bekerja per seri waktu, Anda harus menerapkan ini per komoditi jika Anda memiliki banyak komoditi.
# Jika Anda ingin menerapkan per komoditas, Anda harus mengulanginya:

# Untuk satu komoditi (seperti dalam contoh df saat ini):
df_aktual['Harga Petani '] = stl_imputation(df_aktual['Harga Petani '], period=30) # Asumsi periode mingguan
df_aktual['Harga Pengecer '] = stl_imputation(df_aktual['Harga Pengecer '], period=30)


print("\nDataFrame setelah imputasi:")
print(df_aktual)
print("\nMissing values setelah imputasi:")
print(df_aktual['Harga Petani '].isnull().sum())
print(df_aktual['Harga Pengecer '].isnull().sum())

DataFrame sebelum imputasi:
        Tanggal  Harga Petani   Harga Pengecer 
0    2024-01-01            NaN              NaN
1    2024-01-01            NaN              NaN
2    2024-01-01            NaN              NaN
3    2024-01-01            NaN              NaN
4    2024-01-02        12500.0          13800.0
...         ...            ...              ...
1459 2024-12-30            NaN              NaN
1460 2024-12-31            NaN              NaN
1461 2024-12-31            NaN              NaN
1462 2024-12-31            NaN              NaN
1463 2024-12-31            NaN              NaN

[1464 rows x 3 columns]

Missing values sebelum imputasi:
455
455
Mengisi 455 missing value menggunakan STL imputation...
Pengisian missing value selesai.
Mengisi 455 missing value menggunakan STL imputation...
Pengisian missing value selesai.

DataFrame setelah imputasi:
        Tanggal  Harga Petani   Harga Pengecer 
0    2024-01-01   12625.736548     13781.106934
1    2024-01-01   12430.78

In [160]:
def calculate_mape(actual, predicted):
    actual, predicted = np.array(actual), np.array(predicted)
    return np.mean(np.abs((actual - predicted) / actual)) * 100

In [161]:
df_aktual

,Tanggal,Harga Petani,Harga Pengecer
0,2024-01-01,12625.736548,13781.106934
1,2024-01-01,12430.783214,13686.647524
2,2024-01-01,12467.936563,14042.358097
3,2024-01-01,12558.242739,13760.337283
4,2024-01-02,12500.000000,13800.000000
...,...,...,...
1459,2024-12-30,12908.098000,14806.059814
1460,2024-12-31,12931.576593,14869.377858
1461,2024-12-31,12991.255415,14973.538073
1462,2024-12-31,13057.920810,15118.036751


In [193]:
df_jagung_pred = df[df['Komoditi'] == 'Jagung']
df_kacang_pred = df[df['Komoditi'] == 'Kacang Hijau']
df_medium_pred = df[df['Komoditi'] == 'Beras Medium']
df_premium_pred = df[df['Komoditi'] == 'Beras Premium']

In [174]:
df_pred_list=[
    df_jagung_pred,
    df_kacang_pred,
    df_medium_pred,
    df_premium_pred
]

In [184]:
for i in df_list:
    i=i.set_index('Tanggal')
    i.reset_index(drop=True, inplace=True)

In [197]:
for i in df_pred_list:
    i=i.set_index('Tanggal')
    i=i.reset_index(drop=True, inplace=True)

In [200]:
df_jagung_pred.reset_index(drop=True)

,Tanggal,harga_petani_pred,harga_pengecer_pred,luas_lahan_pred,produktivitas_pred,produksi_pred,hari,bulan,tahun,Komoditi
0,2024-01-01,3943.2832,4356.7603,13853.442,69.503560,163219.50,Monday,January,2024,Jagung
1,2024-01-02,3984.0186,4340.2163,22928.006,69.837260,220739.89,Tuesday,January,2024,Jagung
2,2024-01-03,3977.1821,4308.1920,35320.043,70.248260,305990.53,Wednesday,January,2024,Jagung
3,2024-01-04,3929.3950,4280.3403,45749.656,70.489440,371805.30,Thursday,January,2024,Jagung
4,2024-01-05,3869.6690,4268.3930,52700.310,70.567795,416294.70,Friday,January,2024,Jagung
...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,3628.6238,3924.6770,77579.360,71.467050,573409.60,Friday,December,2024,Jagung
362,2024-12-28,3621.8877,3917.6057,76794.650,71.435520,568265.70,Saturday,December,2024,Jagung
363,2024-12-29,3622.6255,3918.0310,75727.540,71.382614,559863.50,Sunday,December,2024,Jagung
364,2024-12-30,3629.8245,3924.6829,75041.940,71.346380,553816.00,Monday,December,2024,Jagung


In [201]:
df_jagung_x=pd.merge(
    df_jagung,
    df_jagung_pred,
    left_on='Tanggal',
    right_on='Tanggal',
    how='left'
)

In [202]:
df_jagung_x

,Komoditi_x,Tanggal,Harga Petani,Harga Pengecer,harga_petani_pred,harga_pengecer_pred,luas_lahan_pred,produktivitas_pred,produksi_pred,hari,bulan,tahun,Komoditi_y
0,Jagung Pipil Kering,2024-01-01,5679.826328,6008.420521,3943.2832,4356.7603,13853.442,69.503560,163219.50,Monday,January,2024,Jagung
1,Jagung Pipil Kering,2024-01-02,5700.000000,6000.000000,3984.0186,4340.2163,22928.006,69.837260,220739.89,Tuesday,January,2024,Jagung
2,Jagung Pipil Kering,2024-01-03,5700.000000,6000.000000,3977.1821,4308.1920,35320.043,70.248260,305990.53,Wednesday,January,2024,Jagung
3,Jagung Pipil Kering,2024-01-04,5700.000000,6000.000000,3929.3950,4280.3403,45749.656,70.489440,371805.30,Thursday,January,2024,Jagung
4,Jagung Pipil Kering,2024-01-05,5700.000000,6000.000000,3869.6690,4268.3930,52700.310,70.567795,416294.70,Friday,January,2024,Jagung
...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,Jagung Pipil Kering,2024-12-27,3900.000000,4000.000000,3628.6238,3924.6770,77579.360,71.467050,573409.60,Friday,December,2024,Jagung
362,Jagung Pipil Kering,2024-12-28,3900.000000,4000.000000,3621.8877,3917.6057,76794.650,71.435520,568265.70,Saturday,December,2024,Jagung
363,Jagung Pipil Kering,2024-12-29,3900.000000,4000.000000,3622.6255,3918.0310,75727.540,71.382614,559863.50,Sunday,December,2024,Jagung
364,Jagung Pipil Kering,2024-12-30,3891.571104,3989.479703,3629.8245,3924.6829,75041.940,71.346380,553816.00,Monday,December,2024,Jagung


In [203]:
mape_harga = calculate_mape(df_jagung_x['Harga Petani '], df_jagung_x['harga_petani_pred'])
print(f"MAPE untuk harga: {mape_harga:.2f}%")

MAPE untuk harga: 18.85%
